# UVERA · NB03 — Cash-Flow Guardian

| | |
|---|---|
| **Who runs it** | Teammate (P2) |
| **Accelerator** | GPU T4 ×1 |
| **Internet** | On (downloads amazon/chronos-2 from Hugging Face) |
| **Secret** | — |
| **Inputs to add** | — |
| **Expected runtime** | ~20–40 min |
| **How to run** | **Save Version → Save & Run All (Commit)** — runs in the background, no 20-minute idle timeout, 12-hour cap |

**What this AI does (say this to judges):** AI-3 forecasts each customer's money in and out for the next 7 days with an uncertainty band, and estimates the chance their balance falls too low before month-end, so the app can warn early and suggest a safe savings plan. Three forecasters are compared fairly: seasonal-naive, LightGBM-quantile and Chronos-2 (a pretrained time-series foundation model). The winner is chosen on validation weeks.

**Outputs** (in `/kaggle/working/outputs/NB03_outputs.zip`): `artifacts/ai3/` (forecasts.parquet, shortfall.parquet, shortfall_calibrator.json), `reports/metrics_ai3.json`, figures

All data is synthetic or open-licensed. No real customer data.

In [ ]:
# 1) Get the UVERA code from the public GitHub repo (nothing to upload by hand)
REPO = "https://github.com/ABRUBAB/takaresolve-ai-hackathon-2026.git"
REF = "main"   # replace with a commit hash to reproduce an exact run
import os, sys, subprocess
if not os.path.exists("/tmp/uvera"):
    subprocess.run(["git", "clone", "-q", REPO, "/tmp/uvera"], check=True)
subprocess.run(["git", "-C", "/tmp/uvera", "fetch", "-q", "origin", REF], check=False)
subprocess.run(["git", "-C", "/tmp/uvera", "checkout", "-q", REF], check=True)
COMMIT = subprocess.run(["git", "-C", "/tmp/uvera", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.environ["UVERA_ROOT"] = "/tmp/uvera"
sys.path.insert(0, "/tmp/uvera/ml")   # use Kaggle's own numpy/pandas/sklearn/lightgbm (no reinstall)
print("UVERA code at commit", COMMIT)

In [ ]:
!pip install -q "chronos-forecasting>=2.0"

In [ ]:
# 2) Reproducibility: seeds, versions, time budget, output folder
import json
from IPython.display import Image, display
from uvera_ml.common import env_info, set_seed, Budget, out_dir, zip_outputs, write_json
set_seed(42)
budget = Budget(minutes=180)
OUT = out_dir("NB03")
info = env_info(); info["uvera_commit"] = COMMIT
write_json(OUT / "reports" / "versions_nb03.json", info)
print(json.dumps(info, indent=1))

In [ ]:
# 3) Build the synthetic world. Same seed -> identical data on every Kaggle account (checked by hash).
from pathlib import Path
from uvera_ml.sim.world import load_or_generate
world = load_or_generate("/tmp/world_full", scale="full")
print(json.dumps(world.meta, indent=1)[:1500])
ref = Path("/tmp/uvera/reports/world_meta.json")
if ref.exists():
    expected = json.loads(ref.read_text(encoding="utf-8"))["meta"]["hashes"]["events"]
    print("Data identical to NB00 run:", expected == world.meta["hashes"]["events"])
budget.check("world")

## Rolling-origin backtest
Forecasts are made from several start days in the validation and test weeks (never shuffled in time). Metrics: MASE (error vs a naive forecast; < 1 is better), pinball loss, 80% interval coverage (should be close to 0.80). Probabilities are then recalibrated on validation weeks.

In [ ]:
import pandas as pd
from uvera_ml.models.forecasting import run_ai3
summary = run_ai3(world, OUT, n_customers=2000, use_chronos=True)
display(pd.DataFrame(summary["backtest"]))
print("Winner on validation:", summary["winner_on_validation"])
print(json.dumps(summary["shortfall_probability_test"], indent=1, default=float))
display(Image(str(OUT / "reports" / "figures" / "ai3_forecast_example.png")))
budget.check("nb03")

In [ ]:
# Last step: package the results. Download the zip from the Output tab and send it to Rubab.
budget.check("done")
zip_outputs(OUT)